07_door_or_trap

In [ ]:
import re, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR = Path("outputs")
stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl").reset_index(drop=True)
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")
ORDER = ["Traditional", "Gig-proxy", "Startup-proxy"]
d = stage2.copy()

def vlab(series, var):
    """SPSS value labels for variable `var`; missing stays missing."""
    m = {}
    for k, v in VL.get(var, {}).items():
        try: m[str(int(float(k)))] = v
        except Exception: m[str(k)] = v
    s = series.astype(str).str.replace(r"\.0$", "", regex=True)
    return s.map(m).fillna(s).where(series.notna(), np.nan)

daywise = sorted(c for c in d.columns if re.match(r"^b6q\d+_3pt\d+", c))
print("Day-wise wage columns found:", daywise)
display(d[daywise].apply(pd.to_numeric, errors="coerce").describe().round(1))
print("CHECK: these should look like rupee amounts per day (0 to a few thousand).")
print("CHECK: the plan mentions up to 14 columns (two activities). Only the names above exist here.")

for c in ["Total_Income", "Total_income_clean"]:
    x = pd.to_numeric(d[c], errors="coerce")
    print(f"{c}: non-missing {x.notna().mean():.3f} | >0 {(x > 0).mean():.3f} | <0 {(x < 0).mean():.3f} "
          f"| median of positives {x[x > 0].median():.0f}")
for c in ["Casual_Labour", "earning_mem"]:
    print(f"\n{c}:"); print(d[c].value_counts(dropna=False).head(8))
print("\nUsual status codes:"); print(d["usual_status"].value_counts(dropna=False))

In [ ]:
for c in ["Total_Income"] + daywise:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["weekly_casual"] = d[daywise].sum(axis=1, min_count=1)        # rupees earned in the last week
d["casual_monthly_A"] = d["weekly_casual"] * 30 / 7             # assumption A: weekly x 30/7
d["casual_monthly_B"] = d["weekly_casual"] * 4                  # assumption B: weekly x 4

for k in ["A", "B"]:
    d[f"earn_{k}"] = d["Total_Income"].fillna(0) + d[f"casual_monthly_{k}"].fillna(0)
    d.loc[d["Total_Income"].isna() & d["weekly_casual"].isna(), f"earn_{k}"] = np.nan   # nothing recorded

def earn_table(df):
    rows = {}
    for g in ORDER:
        x = df[df["own_economy_type"] == g]
        pos = x.loc[x["earn_A"] > 0, "earn_A"]
        rows[g] = {
            "n": len(x),
            "Total_Income > 0": (x["Total_Income"] > 0).mean(),
            "Total_Income < 0 (losses)": (x["Total_Income"] < 0).mean(),
            "any earnings incl. casual wages (A)": (x["earn_A"] > 0).mean(),
            "any earnings incl. casual wages (B)": (x["earn_B"] > 0).mean(),
            "median earnings of earners, A (Rs/month)": pos.median(),
            "25th percentile": pos.quantile(0.25), "75th percentile": pos.quantile(0.75),
            "median of earners, B (Rs/month)": x.loc[x["earn_B"] > 0, "earn_B"].median(),
        }
    return pd.DataFrame(rows)

earn = earn_table(d)
display(earn.round(3)); earn.round(3).to_csv(OUT_DIR / "dot_earnings.csv")

In [ ]:
d["pc_exp"] = pd.to_numeric(d["b3q5pt6_hhv1"], errors="coerce") / pd.to_numeric(d["b3q1_hhv1"], errors="coerce")
d["pc_exp_quintile"] = pd.qcut(d["pc_exp"], 5, labels=["Q1 (poorest)", "Q2", "Q3", "Q4", "Q5 (richest)"])

q = pd.crosstab(d["pc_exp_quintile"], d["own_economy_type"], normalize="columns")[ORDER].round(3)
print("Share of each economy type's women in each household expenditure quintile (quintiles among working women):")
display(q); q.to_csv(OUT_DIR / "dot_expenditure_quintiles.csv")

q.plot.bar(figsize=(8, 4), color=["#1F3A5F", "#B5483A", "#7F9BB8"])
plt.ylabel("Share within economy type"); plt.title("Household welfare by economy type")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_dot_expenditure.png", dpi=150); plt.show()

In [ ]:
d["ss_label"] = vlab(d["Social_Security"], "Social_Security")
d["contract_label"] = vlab(d["b5pt1q11_perv1"], "b5pt1q11_perv1")
print(d["ss_label"].value_counts(dropna=False)); print(d["contract_label"].value_counts(dropna=False))

lab_ss = d["ss_label"].astype(str).str.lower()
d["ss_recorded"] = d["ss_label"].notna() & ~lab_ss.str.contains("not known")
d["ss_none"] = d["ss_recorded"] & lab_ss.str.contains("none")
lab_c = d["contract_label"].astype(str).str.lower()
d["contract_recorded"] = d["contract_label"].notna()
d["has_written"] = d["contract_recorded"] & ~lab_c.str.contains("no written")
if d["ss_none"].sum() == 0:
    print("WARNING: no 'None' label found for Social_Security. Check the printed labels above and set ss_none by hand.")

In [ ]:
def scorecard(df):
    rows = {}
    for g in ORDER:
        x = df[df["own_economy_type"] == g]
        rows[g] = {
            "n": len(x),
            "rural share": (x["sector"] == "1").mean(),
            "any earnings incl. casual (A)": (x["earn_A"] > 0).mean(),
            "median monthly earnings, earners (A, Rs)": x.loc[x["earn_A"] > 0, "earn_A"].median(),
            "poorest 40% of households": x["pc_exp_quintile"].isin(["Q1 (poorest)", "Q2"]).mean(),
            "social security recorded": x["ss_recorded"].mean(),
            "no social security (of recorded)": x.loc[x["ss_recorded"], "ss_none"].mean() if x["ss_recorded"].any() else np.nan,
            "contract recorded": x["contract_recorded"].mean(),
            "written contract (of recorded)": x.loc[x["contract_recorded"], "has_written"].mean() if x["contract_recorded"].any() else np.nan,
        }
    return pd.DataFrame(rows)

score = scorecard(d)
display(score.round(3)); score.round(4).to_csv(OUT_DIR / "dot_scorecard.csv")

fig, ax = plt.subplots(figsize=(8, 4))
data = [d.loc[(d["own_economy_type"] == g) & (d["earn_A"] > 0), "earn_A"].clip(upper=d["earn_A"].quantile(0.99)) for g in ORDER]
ax.boxplot(data, labels=ORDER, showfliers=False)
ax.set_ylabel("Monthly earnings, earners only (Rs, assumption A)"); ax.set_title("Earnings by economy type")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_dot_earnings.png", dpi=150); plt.show()

In [ ]:
STATUS = {"11": "Own-account worker", "41": "Casual labour, public works", "51": "Casual labour, other"}
d["status_code"] = d["usual_status"].astype(str).str.replace(r"\.0$", "", regex=True)

def diff_ci(k1, n1, k0, n0):
    if n1 == 0 or n0 == 0: return (np.nan,) * 5
    p1, p0 = k1 / n1, k0 / n0
    se = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    return p1, p0, p1 - p0, p1 - p0 - 1.96 * se, p1 - p0 + 1.96 * se

always = lambda x: pd.Series(True, index=x.index)
metrics = {
    "any earnings incl. casual (A)": (lambda x: x["earn_A"] > 0, always),
    "poorest 40% of households": (lambda x: x["pc_exp_quintile"].isin(["Q1 (poorest)", "Q2"]), lambda x: x["pc_exp_quintile"].notna()),
    "no social security | recorded": (lambda x: x["ss_none"], lambda x: x["ss_recorded"]),
    "written contract | recorded": (lambda x: x["has_written"], lambda x: x["contract_recorded"]),
}
rows = []
for code, name in STATUS.items():
    sub = d[d["status_code"] == code]
    gig, trad = sub[sub["own_economy_type"] == "Gig-proxy"], sub[sub["own_economy_type"] == "Traditional"]
    for mname, (num_f, den_f) in metrics.items():
        k1, n1 = int((num_f(gig) & den_f(gig)).sum()), int(den_f(gig).sum())
        k0, n0 = int((num_f(trad) & den_f(trad)).sum()), int(den_f(trad).sum())
        p1, p0, diff, lo, hi = diff_ci(k1, n1, k0, n0)
        rows.append({"status group": name, "measure": mname, "n gig": n1, "n traditional": n0,
                     "gig share": p1, "traditional share": p0, "difference": diff, "95% CI low": lo, "95% CI high": hi})
lfl = pd.DataFrame(rows).round(3)
display(lfl); lfl.to_csv(OUT_DIR / "dot_like_for_like.csv", index=False)

med = (d[d["earn_A"] > 0].loc[d["status_code"].isin(STATUS)]
       .groupby(["status_code", "own_economy_type"])["earn_A"].agg(["size", "median"]).round(0))
print("Median monthly earnings of earners within status groups (assumption A):"); display(med)
print("Rows with CI containing 0 show no clear difference. CIs ignore household clustering, so treat them as rough.")